# Мини-практикум по основам PyTorch

В этом ноутбуке студенты выполняют задания самостоятельно.

Цель практикума:
- научиться работать с тензорами и autograd;
- определить собственную модель на основе `nn.Module`;
- подготовить данные и использовать `DataLoader`;
- написать цикл обучения;
- провести базовую диагностику модели и сравнить CPU/GPU.

> Важное замечание: не используйте готовые решения из кода ниже как "заполненные" ответы. Ниже оставлены только подсказки, заделки и задания для самостоятельного выполнения.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import numpy as np
import time
import os

# При необходимости добавьте сюда другие импорты

## Часть 1. Tensor и autograd

Задание:
1. Создайте тензор `x` с `requires_grad=True`.
2. Постройте простую функцию от `x`, например `y = x^2 + 3*x + 1`.
3. Вычислите скалярную функцию потерь, например `loss = y.sum()`.
4. Вызовите `backward()` и проверьте, что у `x` появился градиент.

Подсказки:
- Для создания тензора можно использовать `torch.tensor(...)` или `torch.randn(...)`.
- После `backward()` посмотрите на `x.grad`.
- Подумайте, как можно проверить, что градиент вычислен правильно.

In [ ]:
# TODO: создайте тензор x с requires_grad=True
# TODO: вычислите простую функцию y от x
# TODO: посчитайте loss и вызовите backward()
# TODO: выведите x.grad и проверьте результат

x = ...
y = ...
loss = ...

loss.backward()

print("x:", x)
print("grad x:", x.grad)

## Часть 2. Модель `nn.Module`

Задание:
1. Определите класс модели, который наследуется от `nn.Module`.
2. Добавьте два линейных слоя (`nn.Linear`).
3. Реализуйте метод `forward`.
4. Создайте пример входа и проверьте форму выхода.

Подсказки:
- Для простого классификатора полезно добавить `ReLU` между слоями.
- Проверьте, что входные и выходные размеры согласованы.
- Для теста используйте `torch.randn(batch_size, input_size)`.

In [ ]:
class SimpleClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        # TODO: создайте два линейных слоя
        # TODO: при необходимости добавьте активацию
        self.fc1 = ...
        self.fc2 = ...

    def forward(self, x):
        # TODO: реализуйте forward
        # TODO: примените слои и активацию
        return ...

# TODO: задайте размеры входа, скрытого слоя и числа классов
INPUT_DIM = ...
HIDDEN_DIM = ...
OUTPUT_DIM = ...

# TODO: создайте модель и проверьте её состояние
model = SimpleClassifier(...)
print(model)

# TODO: создайте dummy_input и проверьте размер выхода
x_test = torch.randn(...)
out = model(x_test)
print("shape:", out.shape)

## Часть 3. Данные: `Dataset` и `DataLoader`

Задание:
1. Создайте собственный класс `CustomDataset`, который хранит признаки и метки.
2. Реализуйте методы `__len__` и `__getitem__`.
3. Подключите `Dataset` к `DataLoader`.
4. Проверьте, что размер батча соответствует ожидаемому, и посмотрите на первую итерацию.

Подсказки:
- Данные можно сделать синтетическими: `X_data` и `Y_data`.
- Используйте `torch.tensor(..., dtype=torch.float32)`.
- В `DataLoader` можно задать `shuffle=True`.

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, features, labels):
        # TODO: сохраните признаки и метки в виде тензоров
        self.features = ...
        self.labels = ...

    def __len__(self):
        # TODO: верните количество элементов
        return ...

    def __getitem__(self, idx):
        # TODO: верните пару (features[idx], labels[idx])
        return ...

# TODO: создайте синтетические данные
NUM_SAMPLES = ...
FEATURE_DIM = ...
BATCH_SIZE = ...

X_data = ...
Y_data = ...

# TODO: создайте dataset и data_loader
# TODO: проверьте длину dataset
# TODO: получите один batch и выведите его размеры

dataset = CustomDataset(...)
print("len(dataset):", len(dataset))

loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

for inputs, targets in loader:
    print("inputs shape:", inputs.shape)
    print("targets shape:", targets.shape)
    break

In [ ]:
# Создайте dataset и dataloader на базе CIFAR-10 для обучения сверточной сети


## Часть 4. Обучение

Задание:
1. Выберите устройство: CPU или GPU.
2. Создайте модель, функцию потерь и оптимизатор.
3. Напишите цикл обучения на несколько эпох.
4. Выведите значение `loss` после каждой эпохи.
5. Сравните время работы на CPU и GPU (если GPU доступен).

Подсказки:
- Проверяйте `torch.cuda.is_available()`.
- Для переноса модели на устройство используйте `.to(device)`.
- В цикле обучения используйте `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()`.
- Для сравнения времени можно использовать `time.time()`.

In [ ]:
# TODO: выберите устройство
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

# TODO: создайте модель, loss и optimizer
model = SimpleClassifier(...).to(DEVICE)
criterion = ...
optimizer = ...

NUM_EPOCHS = ...
loss_history = []

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0.0

    for inputs, targets in loader:
        # TODO: перенесите данные на устройство
        inputs = inputs.to(DEVICE)
        targets = targets.to(DEVICE)

        # TODO: пройдите через модель, посчитайте loss
        outputs = model(inputs)
        loss = criterion(outputs, targets)

        # TODO: очистите градиенты, сделайте backward и step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    loss_history.append(avg_loss)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | loss: {avg_loss:.4f}")

# TODO: сравните скорость работы на CPU и GPU
# Подсказка: создайте две копии модели, проверьте время выполнения на 10 проходах

print("Сравнение CPU/GPU:")

## Часть 5. Диагностика и мониторинг

Задание:
1. Запишите значения `loss` в TensorBoard.
2. Визуализируйте веса или параметры модели.
3. Измерьте время одной эпохи.
4. Проверьте использование памяти GPU.

Подсказки:
- Для TensorBoard используйте `torch.utils.tensorboard.SummaryWriter`.
- Для визуализации параметров можно взять `model.fc1.weight` и вывести её форму или сохранить график.
- Для памяти GPU можно использовать `torch.cuda.memory_allocated()` и `torch.cuda.memory_reserved()`.

In [ ]:
# TODO: создайте папку для логов TensorBoard
log_dir = "./runs/pytorch_mini_practical"
# TODO: инициализируйте SummaryWriter

# TODO: добавьте scalar-лог для loss_history

# TODO: закройте writer

# TODO: замерьте время одной эпохи
# Подсказка: используйте time.time() до и после прохода по loader

# TODO: визуализируйте параметры модели
# Подсказка: возьмите веса первого слоя и выведите их форму или график

# TODO: проверьте использование памяти GPU
# Подсказка: используйте torch.cuda.memory_allocated() и torch.cuda.memory_reserved()

## Вопросы на защиту

1. Как устроен `autograd` и зачем нужен `backward()`?
2. Что происходит в методе `forward()`?
3. Зачем нужен `DataLoader`?
4. Почему в обучении важно следить за `loss` и метриками?
5. Чем отличается выполнение на CPU и GPU?